# Розвідувальний Аналіз Даних (EDA)

>**Що таке EDA?**  
> Розвідувальний аналіз даних (Exploratory Data Analysis) — це процес дослідження набору даних з метою виявлення основних характеристик, закономірностей, аномалій та зв'язків між змінними. EDA передує будь-якому формальному моделюванню і є фундаментом якісного аналізу.

---
## Крок 0: Встановлення та імпорт бібліотек

Перед початком роботи нам потрібно встановити `kagglehub`. Для всіх користувачів Kaggle цей крок є зайвим, оскільки ця бібліотека вбудована в сервіст, проте для користувачів Google Colab чи Jupyter Notebook слід запустити код у клітинці нижче.

In [ ]:
!pip install kagglehub -q

Для виконання основних завдань EDA обов'язково слід використовувати наступний набір бібліотек

| Бібліотека | Призначення |
|------------|-------------|
|pandas** | Робота з табличними даними (DataFrame) |
|numpy** | Математичні операції, масиви |
|matplotlib** | Базова бібліотека для побудови графіків |
|seaborn** | Статистична візуалізація поверх matplotlib |
|kagglehub** | Завантаження датасетів з Kaggle |

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')  # прибирає попередження які псують вигляд ноутбука

# Глобальні налаштування візуалізації
# plt.rcParams задає параметри за замовчуванням для всіх графіків
plt.rcParams['figure.figsize'] = (12, 6)  # Стандартний розмір графіку в дюймах
plt.rcParams['font.size'] = 12             # Базовий розмір шрифту
sns.set_theme(style='whitegrid', palette='muted')  # Тема seaborn

print('Всі бібліотеки успішно імпортовані!')

---
## Крок 1: Завантаження та початковий огляд даних

**Призначення**

Перш ніж аналізувати — потрібно завантажити дані і зрозуміти їх загальну структуру. На цьому кроці ми дізнаємося:
- скільки рядків і стовпців у датасеті
- як виглядають перші/останні записи
- які назви колонок

Датасет містить інформацію проамериканські гірки** з усього світу: назву, розташування, швидкість, висоту, статус, виробника тощо.

In [ ]:
import os

# Завантаження датасету з Kaggle
                                   # автор    # назва набору даних
path = kagglehub.dataset_download('robikscube/rollercoaster-database')
print(f'Шлях до файлів датасету: {path}')

# Перегляд файлів у директорії датасету
files = os.listdir(path)
print('\nФайли в директорії датасету:')
for f in files:
    print(f'  - {f}')

In [ ]:
# Завантажуємо CSV файл з даними
csv_file = "coaster_db.csv"
df = pd.read_csv(os.path.join(path, csv_file))

print(f'Датасет завантажено: "{csv_file}"')
print(f'Розмір: {df.shape[0]} рядків × {df.shape[1]} стовпців')

`head()` повертає перші N рядків (за замовчуванням 5)

Використовуйте це завжди після завантаження — "перший погляд" на дані

In [ ]:
print('Перші 5 рядків датасету:')
df.head()

`tail()` працює аналогічно до `head` проте повертає останні N рядків

Корисно, щоб перевірити, чи дані не обрізані і чи немає "сміттєвих" рядків наприкінці

In [ ]:
print('Останні 5 рядків датасету:')
df.tail()

`df.columns` почертає список усіх колонок. Це є корисним щоб розуміти, з якими змінними будемо працювати

In [ ]:
print('Список усіх колонок:')
df.columns

Більш естетично приємним для виведення у кінцевому ноутбуці є наступний підхід

In [ ]:
for i, col in enumerate(df.columns, 1):
    print(f'  {i:>2}. {col}')

---
## Крок 2: Дослідження структури та типів даних

**Розуміння типів даних критично важливе:**
- `int64`, `float64` — числові: можна рахувати середнє, медіану, будувати гістограми
- `object` — текстові/категоріальні: потребують окремої обробки
- `datetime64` — дати: потребують конвертації з рядка

>Часта проблема:** дата записана як `"2023-01-15"` (тип `object`), а не як справжня дата. Pandas не знатиме, що це дата, поки ви явно не конвертуєте.

`df.info()` — одна з найважливіших команд в EDA. Її слід запускати відразу після завантаження набору даних щоб отримати широку і детальну картину набору даних

Показує: типи даних, кількість non-null значень, використання пам'яті

In [ ]:
df.info()

`nunique()` — кількість унікальних значень у кожній колонці

- Колонки з 1-2 унікальними значеннями — майже константи
- Колонки з 3 - 10 унікальними значення — категоріальні ознаки
- Колонки з = кількості рядків — це ймовірно ID або унікальні назви

In [ ]:
df.nunique()

Тепер застосуємо df.describe() та отримуємо ключові інсайти з даного набору. 

In [ ]:
df.describe().round(2)

### Як читати `df.describe()`

`df.describe()` повертає таблицю, декожен рядок** — це окрема статистична характеристика:

| Показник | Що означає | На що звертати увагу |
|----------|-----------|----------------------|
|count** | Кількість *не-порожніх* значень | Якщо менше загальної кількості рядків — є пропуски |
|mean** | Середнє арифметичне | Чутливе до викидів — якщо відрізняється від median, є перекіс |
|std** | Стандартне відхилення | Показує "розкид" даних навколо середнього. Велике std = великий розкид |
|min** | Мінімальне значення | Перевірте: чи реалістичне? Від'ємний вік чи вага = помилка |
|25%** | 1-й квартиль (Q1) | 25% даних нижче цього значення |
|50%** | Медіана (Q2) | Середина даних. Стійка до викидів, на відміну від mean |
|75%** | 3-й квартиль (Q3) | 75% даних нижче цього значення |
|max** | Максимальне значення | Перевірте: чи реалістичне? Дуже велике max = можливий викид |

**Корисні висновки:**
- Якщо `mean` >> `median` → розподіл скошений вправо (є великі викиди)
- Якщо `mean` << `median` → розподіл скошений вліво
- `IQR = Q3 - Q1` — міжквартильний діапазон, що охоплює середні 50% даних
- Якщо `min` або `max` виглядають нереально → варто дослідити ці рядки

`df.describe` може бути використане для текстових даних також, проте, воно буде набагато менш інформативним. Для того щоб включити текстові об'єкти слід встановити аргумент  `include='object'`
Як трезульт буде повернуто таблицю яка має наступні показники:

- count: не-порожні
- unique: кількість унікальних 
- top: найчастіше значення 
- freq: його частота

In [ ]:
print('Базова описова статистика (категоріальні колонки):')
df.describe(include='object')

In [ ]:
# Розподіл типів даних по датасету
dtype_counts = df.dtypes.value_counts()
print('Розподіл типів даних:')
for dtype, count in dtype_counts.items():
    print(f'  {str(dtype):<12}: {count} колонок')

---
## Крок 3: Аналіз та обробка пропущених значень

**Призначення**

Реальні дані майже завжди містять пропуски (NaN, None). Пропущені значення можуть:
- спотворювати результати аналізу (середнє рахується лише по наявних)
- викликати помилки в алгоритмах машинного навчання
- вказувати на проблеми зі збором даних

**Стратегії обробки:**

| Стратегія | Код | Коли використовувати |
|-----------|-----|----------------------|
| Заповнити середнім | `fillna(df[col].mean())` | Числові, невелика частка пропусків |
| Заповнити медіаною | `fillna(df[col].median())` | Числові, є викиди |
| Заповнити модою | `fillna(df[col].mode()[0])` | Категоріальні |
| Видалити рядки | `dropna(subset=[col])` | Якщо пропуск критичний (напр., цільова змінна) |
| Залишити як є | — | Якщо пропусків < 5% і вони не критичні |

Базовий перегляд кількості пропущених значень можна виконати використовуючи `df.isnull().sum()`, проте для кращого розуміння контексту краще використовувати метод наведений нижче, адже він повертатиме дані у зручному форматі, а також поміщатиме їх у відстотковий контекст пропусків

In [ ]:
# Підрахунок та відсоток пропущених значень
missing = pd.DataFrame({
    'Пропущено (к-сть)': df.isnull().sum(),
    'Пропущено (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['Пропущено (к-сть)'] > 0].sort_values('Пропущено (%)', ascending=False)

print(f'Колонок з пропусками: {len(missing)} з {len(df.columns)}')
print(f'Загальна кількість пропусків: {df.isnull().sum().sum():,}')
missing

Теплова карта пропущених значень використовується в EDA для швидкого аналізу missing values у датасеті.
Кожна клітинка відповідає одному значенню:

* світлий колір → значення є
* червоний → значення пропущене (`NaN`)

Така візуалізація допомагає побачити:

* вертикальні червоні смуги → колонка має багато пропусків
* горизонтальні смуги → окремі рядки погано заповнені
* однакові пропуски в тих самих рядках → можливий зв’язок між ознаками або проблема збору даних

Найчастіше використовується для:

* оцінки якості даних
* пошуку проблем у датасеті
* підготовки даних до preprocessing та ML моделей


In [ ]:
cols_with_missing = missing.index.tolist()
missing_matrix = df[cols_with_missing].isnull()

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    missing_matrix.T,
    cmap='RdYlGn_r',
    cbar_kws={'label': '1 = Пропуск, 0 = Є значення'},
    ax=ax,
    yticklabels=True
)
ax.set_title('Теплова карта пропущених значень\n(червоний = пропуск, зелений = є значення)', 
             fontsize=14, pad=15)
ax.set_xlabel('Індекс рядка')
ax.set_ylabel('Колонки')
plt.tight_layout()
plt.show()

Для унаочнення якості ознак можна також побудувати діаграму наведену нижче. Кожна колонка показує частку пропусків (`NaN`) у відповідній ознаці.

Колір сигналізує про рівень проблеми:

* синій (`<20%`) → пропусків мало, зазвичай можна заповнити
* жовтий (`20–50%`) → суттєва кількість пропусків, потребує аналізу
* червоний (`>50%`) → критично багато пропусків, колонку можуть видалити

Пунктирні вертикальні лінії показують пороги:

* 20% — зона уваги
* 50% — критичний рівень

Такий графік допомагає:

* швидко оцінити якість колонок
* знайти проблемні ознаки
* прийняти рішення щодо імпутації або видалення даних

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

colors = ['#e74c3c' if x > 50 else '#f39c12' if x > 20 else '#3498db' 
          for x in missing['Пропущено (%)']]

bars = ax.barh(missing.index, missing['Пропущено (%)'], color=colors, edgecolor='white')

for bar, val in zip(bars, missing['Пропущено (%)']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)

ax.axvline(x=50, color='red', linestyle='--', alpha=0.7, label='50% поріг (критично)')
ax.axvline(x=20, color='orange', linestyle='--', alpha=0.7, label='20% поріг (увага)')
ax.set_title('Відсоток пропущених значень по колонках', fontsize=14, pad=15)
ax.set_xlabel('Відсоток пропущених значень (%)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Створюємо робочу копію датасету (ВАЖЛИВО: завжди зберігайте оригінал!)
df_clean = df.copy()

key_numeric = ['speed_mph', 'height_ft', 'Inversions_clean', 'Gforce_clean', 'year_introduced']
key_numeric = [c for c in key_numeric if c in df_clean.columns]

# Заповнюємо пропуски медіаною для числових колонок
# Медіана краща за середнє, бо стійка до викидів
for col in ['speed_mph', 'height_ft', 'Gforce_clean']:
    if col in df_clean.columns:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f'{col}: пропуски заповнені медіаною = {median_val:.2f}')


---
## Крок 4: Прості графіки — Bar Chart та Line Chart

Починаємо з найпростіших і найзрозуміліших типів графіків. Вони ідеальні для першого знайомства з даними.

### Bar Chart (стовпчаста/барна діаграма)
-Коли використовувати:** порівняння кількостей або значень між категоріями.  
-Як читати:** висота (або довжина) стовпця пропорційна значенню. Більший стовпець = більше значення.  
-Горизонтальний варіант** (barh) зручніший, коли назви категорій довгі.

### Line Chart (лінійний графік)
-Коли використовувати:** зміна значення з часом (часові ряди).  
-Як читати:** вісь X — час або послідовність, вісь Y — значення. Нахил вгору = зростання, вниз = падіння.

Приклад побудови стовбчастої діаграми наведено нижче

In [ ]:
if 'Type_Main' in df_clean.columns:
    type_counts = df_clean['Type_Main'].value_counts().head(10)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Колір ми обираємо з палітри seaborn
    colors = sns.color_palette('Set2', len(type_counts))
    bars = ax.bar(type_counts.index, type_counts.values, color=colors, edgecolor='white', linewidth=0.8)
    
    # Підписи значень над кожним стовпцем
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')
    
    ax.set_title('Кількість гірок за типом конструкції', fontsize=14, pad=12)
    ax.set_xlabel('Тип гірки')
    ax.set_ylabel('Кількість гірок')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()

Цей графік показує порівняння гірок. З нього чітко зрозуміло що металеві американські гірки є найбільш популярними і явно переважають над іншими варіантами матеріалу конструкції

Нижче наведено приклад використання горизонтального графіку. Його перевагою є зручність показу для довгих назв категорій (у даному випадку — виробників гірок)

In [ ]:
if 'Manufacturer' in df_clean.columns:
    top_manufacturers = df_clean['Manufacturer'].value_counts().head(10)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors = sns.color_palette('Blues_r', len(top_manufacturers))
    bars = ax.barh(top_manufacturers.index, top_manufacturers.values, color=colors, edgecolor='white')

    # Підписи значень справа від кожного бару
    for bar, val in zip(bars, top_manufacturers.values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=10)

    ax.set_title('Топ-10 виробників гірок за кількістю', fontsize=14, pad=12)
    ax.set_xlabel('Кількість гірок')
    ax.invert_yaxis()  # Найбільший — зверху
    plt.tight_layout()
    plt.show()

Тепер розглянемо використання лінійного графіку. Даний графік є дещо комплекснішим ніж попередні, адже поєднує дані по роках з ковзаючим середнім значенням. Використання ковзкого середнього допомоагає зрозуміти більш загальні тренди, прибираючи варіації з шумних одиночних вимірів. Особливо корисним використання ковзких середніх значень є у випадках роботи з даними які часто оновлюються (для прикладу щодня, щогодини). 

Зверніть увагу на те, що графіки можна комбінувати використовуючи `plt.subplot`

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))

yearly = df_clean['year_introduced'].value_counts().sort_index()

# Базова лінія — кількість нових гірок кожного року
ax.plot(yearly.index, yearly.values, color='steelblue', linewidth=1.5, alpha=0.7, label='По роках')

# Ковзне середнє (5 років) — згладжує шум і показує тренд
# rolling(5).mean() = середнє за поточний і 4 попередніх роки
rolling_avg = yearly.rolling(5).mean()
ax.plot(rolling_avg.index, rolling_avg.values, color='red', linewidth=2.5, label='Ковзне середнє (5 років)')

ax.set_title('Кількість нових гірок по роках', fontsize=14, pad=12)
ax.set_xlabel('Рік')
ax.set_ylabel('Кількість нових гірок')
ax.legend()

# Підсвічуємо пік
peak_year = int(yearly.idxmax())
peak_val = int(yearly.max())
ax.annotate(f'Пік: {peak_year}\n({peak_val} гірок)',
            xy=(peak_year, peak_val), xytext=(peak_year - 15, peak_val - 5),
            arrowprops=dict(arrowstyle='->', color='darkred'),
            fontsize=11, color='darkred')

plt.tight_layout()
plt.show()

---
## Крок 5: Розподіли — Histogram та KDE Plot

Тепер досліджуємо,як розподілені** числові змінні. Це одне з ключових завдань EDA.

### Histogram (гістограма)
**Що показує:** як часто зустрічаються значення в різних діапазонах (bins).  
**Як читати:**  
- Вісь X — значення змінної, поділені на рівні інтервали (bins)
- Вісь Y — кількість спостережень у кожному інтервалі  
- Висока смужка = багато значень у цьому діапазоні

**Типи розподілів:**
-Нормальний (дзвоноподібний)** — більшість значень навколо середнього
-Правостороннє скошення (right skew)** — хвіст вправо, є великі викиди
-Лівостороннє скошення (left skew)** — хвіст вліво
-Бімодальний** — два піки, можливо дві різні групи в даних

### KDE Plot (Kernel Density Estimation)
**Що показує:** згладжену криву щільності розподілу.  
**Як читати:** як гістограма, але без "стрибків" між bins. Площа під кривою = 1. Пік = найбільш "типове" значення.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

data = df_clean['speed_mph'].dropna()

# histplot з kde=True малює і гістограму, і KDE-криву одночасно
sns.histplot(data, kde=True, ax=ax, color='steelblue', edgecolor='white', bins=30)

# Додаємо вертикальні лінії середнього і медіани
mean_val = data.mean()
median_val = data.median()
ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Середнє: {mean_val:.1f} mph')
ax.axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Медіана: {median_val:.1f} mph')

ax.set_title('Розподіл швидкості гірок (speed_mph)', fontsize=14, pad=12)
ax.set_xlabel('Швидкість (mph)')
ax.set_ylabel('Кількість гірок')
ax.legend()
plt.tight_layout()
plt.show()


Як можна побачити — дані збалансонвані, адже медіана та середнє значення є майше одним і тим самим числом. Розподіл даних — нормальний з правостороннім скошенням, що видно по довгому хвості (у даному контексті — рекордні швидкості у конкретних парках розваг)

Як було згадано раніше `plt.subplot` дозволяє поєднувати графіки. Часто слід створювати сітки графіків, які демонструють різні ознаки, проте подібні за природою графіки (як графіки розподілу у даному випадку). 

In [ ]:
numeric_cols = ['speed_mph', 'height_ft', 'Inversions_clean', 'Gforce_clean', 'year_introduced']
numeric_cols = [c for c in numeric_cols if c in df_clean.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))  # Задання розміру графіку та розміру сітки
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    data = df_clean[col].dropna()
    sns.histplot(data, kde=True, ax=axes[i], color='steelblue', edgecolor='white')
    axes[i].set_title(f'Розподіл: {col}', fontsize=13)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Кількість')
    axes[i].axvline(data.mean(), color='red', linestyle='--', label=f'Середнє: {data.mean():.1f}')
    axes[i].axvline(data.median(), color='green', linestyle='--', label=f'Медіана: {data.median():.1f}')
    axes[i].legend(fontsize=9)

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Розподіл числових змінних\n(червона пунктирна = середнє, зелена = медіана)', 
             fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## Крок 6: Box Plot та Violin Plot

Ці два типи графіків показуютьрозподіл та розкид** даних, але роблять це по-різному. Обидва широко використовуються в EDA.

### Box Plot (ящик з вусами)

![image.png](https://miro.medium.com/max/9000/1*2c21SkzJMf3frPXPAR_gZA.png)

| Елемент | Що означає |
|---------|------------|
|Коробка (Interquartile range)** | Містить середні 50% даних (від Q1 до Q3) |
|Лінія всередині** | Медіана (Q2) — середина даних |
|Вуса (whiskers)** | Простягаються до `Q1 - 1.5×IQR` (min) та `Q3 + 1.5×IQR` (max) |
|Крапки поза вусами** | Викиди (outliers) — незвичайно великі або малі значення |

**Як порівнювати групи:** якщо коробки не перекриваються → групи статистично відрізняються.

Спершу подивимось на одну змінну (швидкість), щоб зрозуміти всі елементи

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

data = df_clean['speed_mph'].dropna()

bp = ax.boxplot(data, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightsteelblue', color='navy'),
                flierprops=dict(marker='o', color='red', markersize=5, alpha=0.6),
                medianprops=dict(color='red', linewidth=2.5),
                whiskerprops=dict(color='navy'),
                capprops=dict(color='navy'))

# Додаємо анотації для пояснення
q1 = data.quantile(0.25)
q2 = data.median()
q3 = data.quantile(0.75)
ax.annotate(f'Q3 = {q3:.1f}', xy=(1.05, q3), fontsize=10, color='navy')
ax.annotate(f'Медіана = {q2:.1f}', xy=(1.05, q2), fontsize=10, color='red')
ax.annotate(f'Q1 = {q1:.1f}', xy=(1.05, q1), fontsize=10, color='navy')

ax.set_title('Box Plot: Розподіл швидкості гірок\n(червоні крапки = викиди)', fontsize=13, pad=12)
ax.set_ylabel('Швидкість (mph)')
ax.set_xticklabels([])
plt.tight_layout()
plt.show()

#### Читаємо Графік:

*Q1 (Перший квартиль) = 40.0 mph:** 25% усіх американських гірок мають швидкість меншу за це значення.
*Медіана = 49.7 mph:** це «золота середина» набору даних. Рівно половина гірок повільніша за 49.7 mph, а інша половина — швидша.
*Q3 (Третій квартиль) = 55.9 mph:** 75% гірок не перевищують цю швидкість.
*IQR (Міжквартильний розмах) = 15.9 mph:** ширина «коробки» на графіку. Вона охоплює центральні 50% даних і показує, наскільки щільно згрупована основна маса результатів.
*Викиди (Outliers):** точки, що знаходяться вище верхнього «вуса» (зокрема ті, що сягають 100–150 mph) — це справжні гірки-рекордсмени, чиї показники значно вибиваються із загальної статистики.

### Violin Plot

Violin plot це Box plot + форма розподілу (KDE) з обох боків.  
**Ширина "скрипки"** в кожній точці показує, скільки значень там зосереджено.

| Violin Plot | Box Plot |
|-------------|----------|
| Показує форму розподілу | Показує лише ключові квантилі |
| Видно, чи є два піки (бімодальність) | Бімодальність прихована |
| Трохи складніше читати | Простіший та компактніший |

**Коли обирати Violin:** коли важливо побачити форму розподілу, а не лише мінімум/максимум/медіану.

Violin додає форму розподілу — видно, де найбільше "концентрація" значень

In [ ]:
if 'Type_Main' in df_clean.columns:
    # Беремо топ-4 типи для кращої читабельності
    top_types = df_clean['Type_Main'].value_counts().head(4).index
    violin_data = df_clean[df_clean['Type_Main'].isin(top_types)]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # --- Violin Plot ---
    sns.violinplot(
        data=violin_data,
        x='Type_Main',
        y='speed_mph',
        palette='Set2',
        inner='quartile',  # Показує Q1, медіану, Q3 всередині скрипки
        ax=axes[0]
    )
    axes[0].set_title('Violin Plot\n(ширина = щільність значень)', fontsize=13)
    axes[0].set_xlabel('Тип гірки')
    axes[0].set_ylabel('Швидкість (mph)')
    axes[0].tick_params(axis='x', rotation=20)

    # --- Box Plot для порівняння ---
    sns.boxplot(
        data=violin_data,
        x='Type_Main',
        y='speed_mph',
        palette='Set2',
        ax=axes[1]
    )
    axes[1].set_title('Box Plot\n(для порівняння)', fontsize=13)
    axes[1].set_xlabel('Тип гірки')
    axes[1].set_ylabel('Швидкість (mph)')
    axes[1].tick_params(axis='x', rotation=20)

    plt.suptitle('Violin vs Box Plot — порівняння одних даних', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

#### 💡 Читаємо Графік:

*Форма «скрипки» та щільність:** Ширші ділянки графіка на графіку вказують на вищу концентрацію гірок з відповідною швидкістю. Наприклад, для типуSteel** ми бачимо найбільше розширення в районі 40–60 mph.
*Квартилі всередині:** Пунктирні лінії всередині кожної фігури позначають медіану та міжквартильний розмах ($Q1$ та $Q3$), що дозволяє оцінити центр розподілу безпосередньо на графіку.
*Порівняння з Box Plot:** У той час як Box Plot (праворуч на графіку) показує лише статистичні пороги, Violin Plot (ліворуч) розкриває реальнуформу** розподілу. Це дозволяє побачити, чи є розподіл симетричним (подібним до нормального) або має кілька піків (бімодальність).
*Хвости та викиди:** Довгі тонкі «хвости» скрипок (особливо у типуSteel**) тягнуться до високих значень швидкості, візуалізуючи діапазон рідкісних, екстремально швидких гірок-рекордсменів.
*Симетрія:** Відносно симетрична форма для більшості типів на графіку свідчить про те, що швидкості навколо медіани розподілені рівномірно.

---
## Крок 7: Scatter Plot — Залежності між змінними

### Scatter Plot (діаграма розсіювання)

**Що показує:** зв'язок між двома числовими змінними. Кожна точка = один об'єкт (у нас — одна гірка).

**Як читати:**
-Точки йдуть вгору-вправо** → пряма кореляція (більше X → більше Y)
-Точки йдуть вниз-вправо** → обернена кореляція (більше X → менше Y)
-Точки "хмарою"** → зв'язок слабкий або відсутній
-Колір/розмір точок** — можна закодувати третю або четверту змінну

**Корисні доповнення:**
- `hue=` → колір точок за категорією (напр., тип гірки)
- `size=` → розмір точок за числовою змінною
- Лінія тренду (`regplot`) → показує загальний напрямок зв'язку

Перевіримо фізичну закономірність: вища гірка = більша швидкість?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(df_clean['height_ft'], df_clean['speed_mph'],
           alpha=0.5, s=40, color='steelblue', edgecolors='white', linewidth=0.3)

ax.set_xlabel('Висота гірки (футів)', fontsize=12)
ax.set_ylabel('Максимальна швидкість (mph)', fontsize=12)
ax.set_title('Scatter Plot: Висота vs Швидкість', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

# Порахуємо кореляцію
corr = df_clean[['height_ft', 'speed_mph']].corr().iloc[0, 1]
print(f'Кореляція (Pearson): {corr:.3f}')

#### Читаємо графік
* **Коефіцієнт кореляції = 0.403:** це підтверджує помірний позитивний зв'язок між параметрами. Хоча зі зростанням висоти швидкість зазвичай зростає, такий показник свідчить, що висота — не єдиний фактор впливу.
* **Конкретні рекорди:** на графіку можна помітити «короля висоти» — гірку, що сягає майже 400 футів, розвиваючи швидкість 100 mph.
* **Аномальна вертикаль (90–95 футів):** на графіку чітко видно вертикальну лінію точок. Це означає, що багато гірок мають ідентичну висоту, але кардинально різну швидкість (від 5 до майже 130 mph), що вказує на різні інженерні рішення при однаковій висоті.
* **Зона «стандарту»:** основне скупчення даних на графіку знаходиться в межах **20–100 футів** висоти та **20–60 mph** швидкості. Це базові параметри для більшості атракціонів.
* **Екстремальний викид:** найшвидша гірка на графіку можна помітити «короля висоти» — гірку, що сягає майже 400 футів, розвиваючи швидкість 100 mph.
* **Аномальна вертикаль (90–95 футів):** на графіку чітко видно вертикальну лінію точок. Це означає, що багато гірок мають ідентичну висоту, але кардинально різну швидкість (від 5 до майже 130 mph), що вказує на різні інженерні рішення при однаковій висоті.
* **Зона «стандарту»:** основне скупчення даних на графіку знаходиться в межах **20–100 футів** висоти та **20–60 mph** швидкості. Це базові параметри для більшості атракціонів.
* **Екстремальний викид:** найшвидша гірка на графіку розвиває швидкість близько 150 mph. При цьому її висота становить лише ~170 футів, що значно менше за найвищі екземпляри, — ймовірно, тут задіяні системи примусового прискорення.

Додаткові характеристики можуть бути інтегровані використовуючи параметр `hue`. Застосуємо його для попереднього графіку щоб відображати матеріали

In [ ]:
if 'Type_Main' in df_clean.columns:
    fig, ax = plt.subplots(figsize=(12, 7))

    top_types = df_clean['Type_Main'].value_counts().head(4).index
    plot_data = df_clean[df_clean['Type_Main'].isin(top_types)]
    
    # Налаштування колірної палітри hue
    # Це не обов'язковий крок, але дозволяє покращити якість візуалізації
    palette = {
        'Steel': '#3498db',
        'Wood': '#e67e22',
        'Other': '#95a5a6',
    }

    for type_name, group in plot_data.groupby('Type_Main'):
        color = palette.get(type_name, '#cccccc')  # Використання кольору з палітри
        ax.scatter(group['height_ft'], group['speed_mph'],
                   label=type_name, alpha=0.6, s=60,
                   color=color, edgecolors='white', linewidth=0.5)

    ax.set_xlabel('Висота (футів)', fontsize=12)
    ax.set_ylabel('Швидкість (mph)', fontsize=12)
    ax.set_title('Scatter Plot: Висота vs Швидкість (колір = тип гірки)', fontsize=14, pad=12)
    ax.legend(title='Тип', fontsize=11)
    plt.tight_layout()
    plt.show()

При роботі з Scatter plot часто застосовується лінія регресії. Вона корисна для показу залежності між двома числовими змінними. Вона відображає загальний тренд даних: якщо лінія зростає — залежність позитивна, якщо спадає — негативна. Допомагає швидко оцінити силу та напрямок зв’язку між ознаками.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Регресія: висота → швидкість
sns.regplot(data=df_clean, x='height_ft', y='speed_mph',
            ax=axes[0], scatter_kws={'alpha': 0.4, 's': 30},
            line_kws={'color': 'red', 'linewidth': 2})
axes[0].set_title('Висота → Швидкість\n(з лінією тренду)', fontsize=13)
axes[0].set_xlabel('Висота (футів)')
axes[0].set_ylabel('Швидкість (mph)')

# Регресія: кількість інверсій → G-force
if 'Inversions_clean' in df_clean.columns and 'Gforce_clean' in df_clean.columns:
    sns.regplot(data=df_clean, x='Inversions_clean', y='Gforce_clean',
                ax=axes[1], scatter_kws={'alpha': 0.4, 's': 30},
                line_kws={'color': 'red', 'linewidth': 2})
    axes[1].set_title('Інверсії → G-force\n(з лінією тренду)', fontsize=13)
    axes[1].set_xlabel('Кількість інверсій')
    axes[1].set_ylabel('G-force')

plt.suptitle('Scatter + Regression Line (тінь = довірчий інтервал 95%)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

#### Читаємо графіки:

* **Лінія регресії (червона лінія):** на обох панелях вона демонструє загальний напрямок зв'язку між показниками; висхідний нахил свідчить про позитивну залежність.
* **Висота та швидкість:** ліва частина показує крутий нахил лінії тренду, що підтверджує: зі збільшенням висоти очікувана швидкість гірки зростає досить стрімко та передбачувано.
* **Інверсії та G-force:** права частина має значно пологішу лінію тренду; це означає, що додавання нових інверсій лише незначною мірою збільшує перевантаження (G-force), а основні значення концентруються навколо позначки 4G незалежно від кількості петель.
* **Довірчий інтервал (світло-червона тінь):** на ця тінь навколо лінії показує 95% впевненість моделі у знайденому тренді. Розширення тіні на краях (особливо для великої висоти та багатьох інверсій) свідчить про меншу точність прогнозу через невелику кількість екстремальних об'єктів.
* **Відхилення від тренду:** велика кількість точок, що знаходяться далеко від червоної лінії, вказує на високу варіативність даних — лінія показує середню тенденцію, але не гарантує точного значення для кожної окремої гірки.

---
## Крок 8: Трансформація даних

**Призначення** 

Часто дані потребують перетворення перед додатковими крокомами аналізу:

- **Конвертація дат** з рядка у формат datetime — щоб витягати місяць, рік, день тижня
- **Бінаризація** — перетворення числових значень у категорії (напр., "повільна" / "швидка")
- **Логарифмічне перетворення** — для скошених розподілів (зменшує вплив великих викидів)

#### 1. Конвертація дати у формат datetime
Потрібно для подальшої роботи з датами: можна отримувати нові числові дані такі як місяць, рік, сезон, день тижня тощо для подальшої агрегації даних

In [ ]:
if 'opening_date_clean' in df_clean.columns:
    df_clean['opening_date_clean'] = pd.to_datetime(
        df_clean['opening_date_clean'],
        errors='coerce'
    )

    print('Дата відкриття конвертована в datetime')

#### 2. Feature Engineering для дат
Створення нових ознак:
- `opening_month` — місяць відкриття
- `opening_season` — сезон відкриття

In [ ]:


df_clean['opening_month'] = df_clean['opening_date_clean'].dt.month

df_clean['opening_season'] = df_clean['opening_month'].map({
    12: 'Зима', 1: 'Зима', 2: 'Зима',
    3: 'Весна', 4: 'Весна', 5: 'Весна',
    6: 'Літо', 7: 'Літо', 8: 'Літо',
    9: 'Осінь', 10: 'Осінь', 11: 'Осінь'
})

print('Нові колонки: opening_month, opening_season')

#### 3. Категоризація швидкості через pd.cut()
`pd.cut()` ділить числові значення на інтервали (bins) та присвоює кожному інтервалу категорію

In [ ]:
if 'speed_mph' in df_clean.columns:
    df_clean['speed_category'] = pd.cut(
        df_clean['speed_mph'],
        bins=[0, 30, 50, 70, 100, 200],
        labels=[
            'Дуже повільна',
            'Повільна',
            'Середня',
            'Швидка',
            'Екстремальна'
        ],
        right=True
    )

    print(df_clean['speed_category'].value_counts())

#### 4. Логарифмічне перетворення
Використовується для зменшення скошеності розподілу та стабілізації великих значень

```
log1p(x) = log(1 + x)
Безпечне для значень = 0
```

У даному датасеті це перетворення не є необхідним, оскільки розподіли не мають критичної скошеності. Додано переважно у навчальних цілях.

In [ ]:
if 'speed_mph' in df_clean.columns:
    df_clean['log_speed'] = np.log1p(df_clean['speed_mph'])
    df_clean['log_height'] = np.log1p(df_clean['height_ft'])

    print('Створено log_speed та log_height')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_clean['speed_mph'].dropna(), kde=True, ax=axes[0], color='coral', edgecolor='white')
axes[0].set_title('Оригінальний розподіл\nspeed_mph', fontsize=13)
axes[0].set_xlabel('Швидкість (mph)')
axes[0].set_ylabel('Кількість')

sns.histplot(df_clean['log_speed'].dropna(), kde=True, ax=axes[1], color='mediumseagreen', edgecolor='white')
axes[1].set_title('Логарифмічний розподіл\nlog(speed_mph + 1)', fontsize=13)
axes[1].set_xlabel('log(Швидкість + 1)')
axes[1].set_ylabel('Кількість')

plt.suptitle('Ефект логарифмічного перетворення', fontsize=14)
plt.tight_layout()
plt.show()

Нижче наведено використання агрегаційних ознак

In [ ]:
if 'opening_season' in df_clean.columns:
    season_order = ['Весна', 'Літо', 'Осінь', 'Зима']
    season_colors = {'Весна': '#2ecc71', 'Літо': '#f39c12', 'Осінь': '#e67e22', 'Зима': '#3498db'}
    season_counts = df_clean['opening_season'].value_counts().reindex(season_order).dropna()
    
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(season_counts.index, season_counts.values,
                  color=[season_colors[s] for s in season_counts.index], edgecolor='white', linewidth=0.8)
    
    for bar, val in zip(bars, season_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(int(val)), ha='center', fontsize=12, fontweight='bold')
    
    ax.set_title('Відкриття гірок за сезоном\n(витягнуто з дати відкриття)', fontsize=14)
    ax.set_ylabel('Кількість гірок')
    plt.tight_layout()
    plt.show()

---
## Крок 9: Комбіновані та складніші графіки

Після освоєння базових типів — переходимо до потужніших інструментів, що поєднують кілька вимірів.

### Heatmap кореляцій (Кореляційна матриця)
Показує силу та напрямок кореляцій між усіма числовими змінними одночасно.  
**Як читати:**
- Значення від -1 (темно-синій) до +1 (темно-червоний)
- +1 = ідеальна пряма залежність, -1 = ідеальна обернена, 0 = немає зв'язку
- Діагональ завжди = 1 (змінна корелює сама з собою)

In [ ]:
corr_cols = ['speed_mph', 'height_ft', 'Inversions_clean', 'Gforce_clean', 'year_introduced']
corr_cols = [c for c in corr_cols if c in df_clean.columns]

corr_matrix = df_clean[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))

# mask приховує верхній трикутник (він дублює нижній)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    annot=True,        # Показати числа в клітинках
    fmt='.2f',         # Формат: 2 знаки після коми
    cmap='RdBu_r',     # Палітра: синій (від'ємна) → білий (0) → червоний (додатна)
    center=0,
    vmin=-1, vmax=1,
    mask=mask,
    ax=ax,
    square=True,
)
ax.set_title('Кореляційна матриця числових змінних', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

#### Читаємо графік:

* **Найсильніший зв'язок (0.41):** спостерігається між висотою (`height_ft`) та швидкістю (`speed_mph`). Це найбільш значуща позитивна кореляція в наборі даних, що підтверджує пряму залежність швидкості від висоти конструкції.
* **Вплив інверсій:** кількість інверсій має слабкий позитивний зв'язок зі швидкістю (0.25) та перевантаженням (0.19), як показано на . Цікаво, що зв'язок інверсій із висотою майже відсутній (0.05), що вказує на можливість проектування петель навіть на невисоких гірках.
* **Еволюція з часом:** показник `year_introduced` має слабку позитивну кореляцію з кількістю інверсій (0.23) та швидкістю (0.15). це натякає на те, що сучасні гірки стають складнішими та швидшими, але прогрес відбувається поступово.
* **Відсутність зв'язку:** найменша кореляція на  зафіксована між роком появи та перевантаженням (-0.04). Це означає, що рівень G-force залишається стабільним протягом десятиліть і не залежить від сучасних технологій, оскільки обмежений фізіологічними можливостями людини.
* **Загальна тенденція:** оскільки всі значущі коефіцієнти на  мають теплий відтінок (позитивні значення), ми бачимо, що покращення одного технічного параметра зазвичай веде до помірного зростання інших.

### Pairplot
Матриця всіх можливих scatter plots між змінними. На діагоналі — KDE розподіл кожної змінної.

In [ ]:
# PAIRPLOT — матриця scatter plots для всіх пар змінних
# Корисно для швидкого огляду всіх залежностей між числовими змінними
# Увага: на великих датасетах може виконуватись довго! Тому беремо sample

pairplot_cols = ['speed_mph', 'height_ft', 'Inversions_clean', 'Gforce_clean']
pairplot_cols = [c for c in pairplot_cols if c in df_clean.columns]

if 'Type_Main' in df_clean.columns:
    top2 = df_clean['Type_Main'].value_counts().head(2).index
    pair_data = df_clean[df_clean['Type_Main'].isin(top2)][pairplot_cols + ['Type_Main']].dropna()
    pair_data_sample = pair_data.sample(min(300, len(pair_data)), random_state=42)

    g = sns.pairplot(pair_data_sample, hue='Type_Main', diag_kind='kde',
                     plot_kws={'alpha': 0.5, 's': 30}, palette='husl')
    g.figure.suptitle('Pairplot: взаємозалежності між числовими змінними\n(два найпопулярніших типи)', 
                      y=1.02, fontsize=13)
    plt.show()

#### Читаємо графік:
* **Порівняння типів (Steel vs Wood):** чітко видно, що сталеві гірки (рожевий колір) домінують у високих діапазонах швидкості та висоти, тоді як дерев'яні (бірюзовий колір) мають значно вужчий та стабільніший розподіл параметрів.
* **Розподіл інверсій:** вертикальна бірюзова лінія в колонці `Inversions_clean` на свідчить про те, що дерев'яні гірки майже ніколи не мають інверсій (їхня кількість зазвичай дорівнює нулю), на відміну від сталевих, де цей показник варіюється від 0 до 14.
* **Концентрація G-force:** більшість значень перевантаження для обох типів на фокусується навколо позначки 4G, що відображено у високих піках на діагональних графіках розподілу.
* **Кореляція швидкості та висоти:** на перетині `speed_mph` та `height_ft` у спостерігається найчіткіша лінійна залежність — чим вища конструкція, тим більша швидкість, причому сталеві гірки значно частіше стають рекордсменами в обох категоріях.
* **Взаємодія інверсій та перевантажень:** розсіяні точки на перетині `Inversions_clean` та `Gforce_clean` у показують, що наявність великої кількості петель не обов'язково призводить до пропорційного зростання G-force; інженерний дизайн дозволяє тримати навантаження в безпечних межах.

---
## Бонус: Корисні однорядкові функції для EDA

Ці функції — маленькі помічники, що економлять час. Зберіть їх у власну бібліотеку!  
Більшість з них написані як `lambda` — анонімні функції для швидкого виклику.

In [ ]:
# ============================================================
# КОРИСНІ ОДНОРЯДКОВІ ФУНКЦІЇ ДЛЯ EDA
# ============================================================

# --- ОГЛЯД ДАНИХ ---

# Швидкий огляд датасету: розмір + типи + пропуски
quick_look = lambda df: print(
    f"Розмір: {df.shape} | Типи: {dict(df.dtypes.value_counts())} | Пропуски: {df.isnull().sum().sum()}"
)

# Відсоток пропущених значень (тільки колонки з пропусками)
missing_pct = lambda df: (
    df.isnull().mean() * 100
).round(2).sort_values(ascending=False)[lambda x: x > 0]

# Кількість дублікатів у датафреймі
count_dupes = lambda df: df.duplicated().sum()

# Частка кожної категорії у колонці (у %)
value_pct = lambda df, col: (df[col].value_counts(normalize=True) * 100).round(2)


# --- ЧИСЛОВІ ДАНІ ---

# Коефіцієнт варіації (чим менше — тим стабільніші дані)
cv = lambda series: round(series.std() / series.mean() * 100, 2)

# Перцентилі: 1%, 5%, 25%, 50%, 75%, 95%, 99%
percentiles = lambda series: series.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

# Нижня та верхня межа викидів (IQR метод)
iqr_bounds = lambda s: (
    s.quantile(0.25) - 1.5 * (s.quantile(0.75) - s.quantile(0.25)),
    s.quantile(0.75) + 1.5 * (s.quantile(0.75) - s.quantile(0.25))
)

# Кількість викидів у колонці (IQR)
count_outliers = lambda s: (
    (s < s.quantile(0.25) - 1.5 * (s.quantile(0.75) - s.quantile(0.25))) |
    (s > s.quantile(0.75) + 1.5 * (s.quantile(0.75) - s.quantile(0.25)))
).sum()

# Нормалізація в діапазон [0, 1] (min-max scaling)
normalize = lambda series: (series - series.min()) / (series.max() - series.min())

# Стандартизація (z-score: mean=0, std=1)
standardize = lambda series: (series - series.mean()) / series.std()


# --- ФІЛЬТРАЦІЯ ---

# Рядки з екстремальними значеннями (|z-score| > N стандартних відхилень)
extreme_rows = lambda df, col, n=3: df[
    df[col].apply(lambda x: abs((x - df[col].mean()) / df[col].std()) > n)
]

# Вибір лише числових колонок
num_cols = lambda df: df.select_dtypes(include='number').columns.tolist()

# Вибір лише категоріальних колонок
cat_cols_fn = lambda df: df.select_dtypes(include='object').columns.tolist()

# Колонки з пропусками (список)
cols_with_na = lambda df: df.columns[df.isnull().any()].tolist()


# --- КОРЕЛЯЦІЇ ---

# Топ-N пар за кореляцією (без самокореляцій та дублікатів)
top_corr = lambda df, n=10: (
    df.select_dtypes('number').corr().abs()
    .unstack().sort_values(ascending=False)
    .drop_duplicates()[lambda x: x < 1].head(n)
)

print('Всі допоміжні функції визначено!')
print('\nПриклади використання:')

In [ ]:
# Демонстрація використання функцій

print('=== quick_look(df_clean) ===')
quick_look(df_clean)

print('\n=== missing_pct(df_clean) — перші 5 ===')
print(missing_pct(df_clean).head())

print('\n=== num_cols(df_clean) ===')
print(num_cols(df_clean))

print('\n=== count_outliers для speed_mph ===')
print(f'Викидів у speed_mph: {count_outliers(df_clean["speed_mph"])}')

print('\n=== top_corr(df_clean) — топ-5 кореляцій ===')
print(top_corr(df_clean, n=5))

print('\n=== percentiles для speed_mph ===')
print(percentiles(df_clean['speed_mph']).round(1))